<a href="https://colab.research.google.com/github/asmaalgarably/-My-portfolio/blob/main/LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain-openai
!pip install -U langchain-tavily

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.4/146.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.8/573.8 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.0/869.0 kB 29.7 MB/s eta 0:00:00
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.14.2
    Uninstalling langsmith-0.14.2:
      Successfully uninstalled langsmith-0.14.2
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.6
    Uninstalling langchain-core-1.6.6:
      Successfully uninstalled langchain-core-1.6.6


In [21]:
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_tavily import TavilySearch
from email.message import EmailMessage
import smtplib
import os
import requests
from google.colab import userdata


In [22]:
llm=ChatOpenAI(
    api_key=userdata.get('OPENROUTER_API_KEY'),
    base_url="https://openrouter.ai/api/v1",
    model_name="openai/gpt-4o-mini"
)
os.environ['TAVILY_API_KEY']=userdata.get('TAVILY_API_KEY')

In [23]:
@tool
def get_serch(query:str)->str:
  "search web about something"
  search_instance=TavilySearch(max_results=5,topic="general",search_depth="advanced")
  res=search_instance.invoke(query)
  return "\n\n".join(f"{r['title']}\n{r['content']}\n{r['url']}" for r in res['results'])

@tool
def send_email(rec_email:str, subject:str, msg:str):
  "This tool to send a email to a person"
  sender_email="asmaalgarably@gmail.com"
  m = EmailMessage()
  m["Subject"] = subject
  m["From"] = sender_email
  m["To"] = rec_email
  m.set_content(msg)
  server = smtplib.SMTP('smtp.gmail.com',587)
  server.starttls()
  server.login(sender_email, userdata.get('GMAIL_APP_PASSWORD'))
  server.send_message(m)
  server.quit()
  return f"Email sent to {rec_email} with subject '{subject}'"
@tool
def send_telegram(message: str) -> str:
  """This tool sends a text message to a designated Telegram channel or user."""
  user_id = "1382515532"
  bot_token = userdata.get('TELEGRAM_BOT_TOKEN')
  url = f"https://api.telegram.org/bot{bot_token}/sendMessage"
  payload = {
      "chat_id": user_id,
      "text": message
  }
  response = requests.post(url, json=payload)
  if response.status_code == 200:
      return "Message sent to Telegram successfully."
  else:
      return f"Failed to send message: {response.text}"

In [24]:
user_question = input("Ask AnyThing ?")
serach = get_serch.invoke(user_question)


agent_email = create_agent(model=llm, tools=[send_email], system_prompt="you are helpful assistant")
res_email = agent_email.invoke({
    "messages": [
        {
            "role": "user",
            "content": f"Forward the message you receive from the agent: {serach} to this email address asmaalgarably@gmail.com."
        }
    ]
})


agent_tel = create_agent(model=llm, tools=[send_telegram], system_prompt="You are a translator assistant. Always translate the user's content into Arabic first, "
        "then call send_telegram ONCE with the full Arabic translation as the message. "
        "Never send English text. Keep it clear and well organized.")
res_tel = agent_tel.invoke({
    "messages": [
        {
            "role": "user",
            "content": f"Translate this into Arabic and send it via Telegram:\n\n{serach}"
        }
    ]
})

Ask AnyThing ?what is the rag system
